## 1. IMPORTING ALL REQUIRE LIBRARIES

In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np 
import sklearn
import joblib

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from feature_engine.datetime import DatetimeFeatures

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
sklearn.set_config(transform_output="default")

In [ ]:
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')
val_df = pd.read_csv('data/val.csv')

In [ ]:
train_df.head()

In [ ]:
def split_data(data):
    x = data.drop(columns='price')
    y = data['price']
    return x, y

In [ ]:
x_train, y_train = split_data(train_df)


In [ ]:
x_train.head()

In [ ]:
y_train.head()

In [ ]:
x_val, y_val = split_data(val_df)
x_test, y_test = split_data(test_df)

In [ ]:
print("SHAPE TRAIN:-",x_val.shape, y_val.shape)

In [ ]:
x_train.info()

### DATA PREPROCESSING :-

In [ ]:
dt_cols = ['date_of_journey', 'dep_time', 'arrival_time']
num_cols = ['duration', 'total_stops']
cat_cols = [col for col in x_train.columns if (col not in dt_cols) and (col not in num_cols)]

In [ ]:
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scalar", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

doj_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("extractor", DatetimeFeatures(features_to_extract=['month', 'week', 'day_of_week', 'day_of_month'], format="mixed")),
    ("scalar", StandardScaler())
])

time_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("extractor", DatetimeFeatures(features_to_extract=['hour', 'minute'], format="mixed")),
    ("scalar", StandardScaler())
])

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols),
    ("doj", doj_transformer, ['date_of_journey']),
    ("time", time_transformer, ['dep_time', 'arrival_time'])
])

In [ ]:
train_df.columns

In [ ]:
preprocessor.fit_transform(x_train)

### MODEL SELECTION

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import r2_score
from sklearn.model_selection import learning_curve

In [ ]:
algorithm = {
    "Linear Regression": LinearRegression(),
    "Support Vector Machine": SVR(),
    "Random Forest": RandomForestRegressor(n_estimators=10),
    "XG Boost": XGBRegressor(n_estimators = 10)
}

In [ ]:
data = pd.concat([train_df, val_df], axis=0)
X_data, y_data = split_data(data)

In [ ]:
def plot_curves(sizes, mean_scores, std_scores, label, ax):
    ax.plot(
                sizes, 
                mean_scores,
                marker = "o",
                label = label
    )
    
    ax.fill_between(
                x = sizes,
                y1 = mean_scores - std_scores,
                y2 = mean_scores + std_scores,
                alpha = 0.5
    )

In [ ]:
def plot_learning_curves(name, algorithm, figsize = (12, 4)):
    model = Pipeline(steps=[
                    ("pre", preprocessor),
                    ("alg", algorithm)
    ])
    
    train_size, train_scores, test_scores = learning_curve(
                    estimator=model,
                    X = X_data, 
                    y = y_data,
                    cv = 3, 
                    scoring = "r2",
                    n_jobs = -1, 
                    random_state = 42
    )
    
    mean_train_scores = np.mean(train_scores, axis = 1)
    std_train_scores = np.std(train_scores, axis = 1)
    train_score = f"{mean_train_scores[-1]:.2f} +/- {std_train_scores[-1]:.2f}"
    
    mean_test_scores = np.mean(test_scores, axis = 1)
    std_test_scores = np.std(test_scores, axis = 1)
    test_score = f"{mean_test_scores[-1]:.2f} +/- {mean_test_scores[-1]:.2f}"
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # for train curve
    plot_curves(
                    train_size,
                    mean_train_scores,
                    std_train_scores,
                    f"Train ({train_score})",
                    ax
    )
    
    # for test curve
    plot_curves(
                    train_size,
                    mean_test_scores,
                    std_test_scores,
                    f"Test ({test_score})",
                    ax
    )
    
    ax.set(xlabel = "Training set size", ylabel = "R-square", title = name)
    ax.legend(loc = "lower right")
    plt.show()

In [ ]:
for name, alg in algorithm.items():
    plot_learning_curves(name, alg)

### MODEL TRAINING 

In [ ]:
model = Pipeline(steps=[
    ("pre", preprocessor),
    ("XG Boost", XGBRegressor(n_estimators = 10))
])


In [ ]:
model.fit(X_data, y_data)

In [ ]:
def evaluate_model(X, y):
    y_pred = model.predict(X)
    return r2_score(y, y_pred)

In [ ]:
evaluate_model(X_data, y_data)

In [ ]:
evaluate_model(x_test, y_test)

In [ ]:
evaluate_model(x_val, y_val)

### 8. MODEL PRESISTANCE

In [ ]:
joblib.dump(model, "model.joblib")

In [ ]:
saved_model = joblib.load('model.joblib')

In [ ]:
saved_model